# Sign language translation demo

Run on Google Colab with trained checkpoints and their matching `spm.model` in Google Drive. The Gradio app compares XE and final SCST translations. See [demo instructions](../demo/README.md) for the expected folder layout.


## 1. Install dependencies


In [ ]:
!pip uninstall -y -q mediapipe mediapipe-nightly 2>/dev/null
!pip install -q "mediapipe==0.10.14" || pip install -q "mediapipe<=0.10.21"
!pip install -q sentencepiece gradio opencv-python-headless

import mediapipe, sys
print("mediapipe", mediapipe.__version__, "| python", sys.version.split()[0])
try:
    mediapipe.solutions.holistic
    print("solutions.holistic OK")
except AttributeError:
    print("top-level solutions thiếu - slt_demo.py sẽ tự import submodule, không sao")


## 2. Load the repository


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/linhxm/Sign-Language-REL.git"
CODE_DIR = Path("/content/Sign-Language-REL")
if not CODE_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(CODE_DIR)], check=True)
assert (CODE_DIR / "main.py").is_file(), f"Invalid repository: {CODE_DIR}"
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
os.environ["SLT_CODE_DIR"] = str(CODE_DIR)
print("Repository:", CODE_DIR)


## 3. Mount Drive and set checkpoint paths


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/slt_demo"   # <<< SỬA CHỖ NÀY
RESULTS_DIR = DRIVE_DIR                          # thư mục cha chứa các run
SPM_PATH = os.path.join(DRIVE_DIR, "spm.model")

assert os.path.isdir(DRIVE_DIR), f"Không thấy {DRIVE_DIR} - kiểm tra lại đường dẫn trên Drive"
print(sorted(os.listdir(DRIVE_DIR)))


## 4. Load the demo modules


In [ ]:
DEMO_DIR = CODE_DIR / "demo"
assert (DEMO_DIR / "app.py").is_file(), f"Missing demo code: {DEMO_DIR}"
sys.path.insert(0, str(DEMO_DIR))


## 5. Check the available checkpoints


In [ ]:
import torch
from slt_demo import discover_runs, Translator

runs = discover_runs(RESULTS_DIR)
assert runs, f"Không thấy best_xe.pt nào dưới {RESULTS_DIR}"
for name, paths in runs.items():
    print(name, "->", {k: os.path.basename(v) for k, v in paths.items()})

RUN = sorted(runs)[0]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
xe = Translator.from_checkpoint(runs[RUN]["xe"], SPM_PATH, device=DEVICE, name="XE")
print(f"\n[{RUN}] nạp OK · device={DEVICE} · vocab={xe.tokenizer.vocab_size} · meta={xe.meta}")


## 6. Launch the interface

`share=True` creates a public Gradio link for the running session.


In [ ]:
import app as demo_app

demo_app.STATE["runs"] = runs
demo_app.STATE["spm"] = SPM_PATH
demo_app.STATE["device"] = DEVICE

demo_app.build_ui().launch(share=True, debug=False)


## 7. Optional: translate without the interface


In [ ]:
from slt_demo import PoseExtractor, Translator

VIDEO = "/content/test.mp4"   # <<< đường dẫn video của bạn

pose, stats, _ = PoseExtractor().extract(VIDEO)
print(f"pose {pose.shape} · detect tay trái/phải: "
      f"{stats.det_lhand*100:.0f}% / {stats.det_rhand*100:.0f}% · {stats.extract_s:.1f}s\n")

for label, key in [("XE  ", "xe"), ("SCST", "scst")]:
    if key not in runs[RUN]:
        continue
    tr = Translator.from_checkpoint(runs[RUN][key], SPM_PATH, device=DEVICE, name=label)
    text, dt = tr.translate(pose, decode="beam", beam_size=4)
    print(f"{label}: {text}   ({dt*1000:.0f} ms)")
